In [1]:
import asyncio
import glob
import httpx
import logging
import pandas as pd
import polars as pl
from datetime import datetime, timezone
from dependency_metrics.analyzer import DependencyAnalyzer
from packaging import version as pkg_version
from packaging.requirements import Requirement
from packaging.specifiers import SpecifierSet
from pathlib import Path
from typing import Any, Optional, Union

In [2]:
# Configure logging
logging.basicConfig(
    level=logging.WARNING,
    format='%(levelname)s: %(message)s'
)
logger = logging.getLogger(__name__)
logging.getLogger("httpx").setLevel(logging.WARNING)


## Import Deps.Dev Data

In [3]:
path_pattern = "/workspaces/mads-siads-699-winter-2026-capstone/notebooks/data/source_data/initial_dataset/*.parquet"

In [4]:
all_files = sorted(glob.glob(path_pattern))
from_file = 6
to_file = 30
if all_files:
    df_deps = (
        pl.scan_parquet(all_files[from_file:to_file], n_rows=10000000)
        .filter(pl.col("package_published_at").is_not_null())
        .collect()
    )
    
    print(f"Finished loading and filtering {len(all_files[from_file:to_file])} files.")

Finished loading and filtering 24 files.


In [5]:
df_deps.head(30)

SnapshotAt,package_name,package_version,package_published_at,dep_name,dep_version,MinimumDepth,github_repo
datetime[μs],str,str,datetime[μs],str,str,i64,str
2023-07-31 21:01:30.722347,"""pyfileconf""","""0.15.2""",2020-08-06 08:12:18,"""bibtexparser""","""1.4.0""",3,"""nickderobertis/py-file-conf"""
2023-07-31 21:01:30.722347,"""pyfileconf""","""0.15.2""",2020-08-06 08:12:18,"""black""","""23.7.0""",1,"""nickderobertis/py-file-conf"""
2023-07-31 21:01:30.722347,"""pyfileconf""","""0.15.2""",2020-08-06 08:12:18,"""click""","""8.1.6""",2,"""nickderobertis/py-file-conf"""
2023-07-31 21:01:30.722347,"""pyfileconf""","""0.15.2""",2020-08-06 08:12:18,"""contourpy""","""1.1.0""",3,"""nickderobertis/py-file-conf"""
2023-07-31 21:01:30.722347,"""pyfileconf""","""0.15.2""",2020-08-06 08:12:18,"""cycler""","""0.11.0""",3,"""nickderobertis/py-file-conf"""
…,…,…,…,…,…,…,…
2023-07-31 21:01:30.722347,"""pyfileconf""","""0.15.2""",2020-08-06 08:12:18,"""matplotlib-inline""","""0.1.6""",2,"""nickderobertis/py-file-conf"""
2023-07-31 21:01:30.722347,"""pyfileconf""","""0.15.2""",2020-08-06 08:12:18,"""mixins""","""0.1.4""",1,"""nickderobertis/py-file-conf"""
2023-07-31 21:01:30.722347,"""pyfileconf""","""0.15.2""",2020-08-06 08:12:18,"""mpmath""","""1.3.0""",3,"""nickderobertis/py-file-conf"""


## Get snapshots of each version
Get the dates (start and end) between each version from the deps.dev table using windowing

In [6]:
date_col = "package_published_at"

snapshots = (
    df_deps.select(["package_name", date_col])
    .unique()
    .sort(["package_name", date_col])
)

snapshots_with_bounds = snapshots.with_columns([
    pl.col(date_col).shift(-1).over("package_name").alias("next_snapshot")
])

snapshots_with_bounds = snapshots_with_bounds.filter(pl.col("next_snapshot").is_not_null())

In [7]:
snapshots_with_bounds.head()

package_name,package_published_at,next_snapshot
str,datetime[μs],datetime[μs]
"""2captcha-python""",2023-06-28 15:01:24,2023-10-09 11:35:45
"""a-cv-imwrite-imread-plus""",2022-12-29 00:56:02,2023-08-14 17:50:48
"""a-pandas-ex-bs4df""",2022-10-29 21:42:52,2023-10-12 17:14:11
"""a-pandas-ex-bs4df-lite""",2023-02-23 18:24:59,2023-10-12 17:18:01
"""a-pandas-ex-image-tools""",2022-12-27 11:12:12,2023-09-05 21:20:06


## Test Zahan et. Al method of getting TTU / TTR

In [ ]:
all_results = []

for row in snapshots_with_bounds.iter_rows(named=True):
    pkg = row['package_name']
    start = row[date_col]
    end = row['next_snapshot']
    
    print(f"Analyzing {pkg} snapshot window: {start} -> {end}")
    
    try:
        analyzer = DependencyAnalyzer(
            ecosystem="pypi",
            package=pkg,
            start_date=start,
            end_date=end,
            weighting_type="exponential",
            half_life=180,
            output_dir=Path("./output")
        )

        results = analyzer.analyze()
        
        all_results.append({
            'package_name': pkg,
            'snapshot_start': start,
            'snapshot_end': end,
            'avg_ttu': results.get('ttu'),
            'avg_ttr': results.get('ttr')
        })
    except Exception as e:
        print(f"Error processing {pkg} at {start}: {e}")

results_df = pd.DataFrame(all_results)

Analyzing 2captcha-python snapshot window: 2023-06-28 15:01:24 -> 2023-10-09 11:35:45
Analyzing a-cv-imwrite-imread-plus snapshot window: 2022-12-29 00:56:02 -> 2023-08-14 17:50:48
Analyzing a-pandas-ex-bs4df snapshot window: 2022-10-29 21:42:52 -> 2023-10-12 17:14:11
Analyzing a-pandas-ex-bs4df-lite snapshot window: 2023-02-23 18:24:59 -> 2023-10-12 17:18:01
Analyzing a-pandas-ex-image-tools snapshot window: 2022-12-27 11:12:12 -> 2023-09-05 21:20:06
Analyzing a10sa-script snapshot window: 2022-09-05 09:16:29 -> 2023-10-11 12:36:54
Analyzing a2conf snapshot window: 2023-03-31 21:35:47 -> 2023-09-30 17:36:12
Analyzing aafitrans snapshot window: 2023-06-04 13:16:09 -> 2023-10-18 06:59:38
Analyzing aamp snapshot window: 2019-04-22 22:37:52 -> 2020-01-17 20:37:29
Analyzing aardwolf snapshot window: 2023-03-02 22:58:33 -> 2023-09-20 22:07:25
Analyzing abb-robot-client snapshot window: 2022-12-28 01:16:27 -> 2023-10-18 00:11:35
Analyzing abcvoting snapshot window: 2022-11-30 13:39:56 -> 202

In [ ]:
results_df.head()

""


## Try to replicate the Zahan et. Al method of getting TTU/TTR, but in this notebook instead of using dependency-metrics
The dependency-metrics forces you to save all of the metadata locally, which we don't want while we are running this analysis. 

In [ ]:
# Constants for connection pooling
LIMITS = httpx.Limits(max_connections=100, max_keepalive_connections=20)
CLIENT = httpx.AsyncClient(limits=LIMITS, timeout=10.0)

# Type Aliases
MetadataDict = dict[str, Any]
VulnerabilityList = list[dict[str, Any]]

# In-memory caches
METADATA_CACHE: dict[str, MetadataDict] = {}
VULN_CACHE: dict[str, VulnerabilityList] = {}


def ensure_utc(dt_val: Union[str, datetime, None]) -> Optional[datetime]:
    """
    Ensure a datetime object is UTC aware.

    :param dt_val: The datetime object or ISO string to convert.
    :return: A UTC-aware datetime object or None if input is None.
    """
    if dt_val is None:
        return None
    if isinstance(dt_val, str):
        # Handle 'Z' suffix and parse to datetime
        dt_val = datetime.fromisoformat(dt_val.replace("Z", "+00:00"))
    if dt_val.tzinfo is None:
        return dt_val.replace(tzinfo=timezone.utc)
    return dt_val.astimezone(timezone.utc)


async def fetch_pypi_metadata(package_name: str) -> Optional[MetadataDict]:
    """
    Fetch and cache metadata for a package from PyPI.

    :param package_name: Name of the PyPI package.
    :return: Dictionary of package metadata or None if request fails.
    """
    if package_name in METADATA_CACHE:
        return METADATA_CACHE[package_name]

    url = f"https://pypi.org/pypi/{package_name}/json"
    try:
        response = await CLIENT.get(url)
        if response.status_code == 200:
            data = response.json()
            processed_releases: list[tuple[pkg_version.Version, Optional[datetime], str]] = []

            for ver_str, files in data.get('releases', {}).items():
                if files:
                    try:
                        ver_obj = pkg_version.parse(ver_str)
                        # Use the first file's upload time as release date
                        rel_date = ensure_utc(files[0].get('upload_time'))
                        processed_releases.append((ver_obj, rel_date, ver_str))
                    except (pkg_version.InvalidVersion, ValueError):
                        continue

            # Sort releases by version object for later traversal
            data['processed_releases'] = sorted(processed_releases, key=lambda x: x[0])
            METADATA_CACHE[package_name] = data
            return data
    except httpx.RequestError as exc:
        logger.error("Failed to fetch metadata for %s: %s", package_name, exc)

    return None


async def get_vulnerabilities_osv(package_name: str, version: str) -> VulnerabilityList:
    """
    Query the OSV database for vulnerabilities for a specific package version.

    :param package_name: The name of the package.
    :param version: The specific version string.
    :return: A list of vulnerability objects.
    """
    cache_key = f"{package_name}@{version}"
    if cache_key in VULN_CACHE:
        return VULN_CACHE[cache_key]

    url = "https://api.osv.dev/v1/query"
    query = {
        "version": version,
        "package": {"name": package_name, "ecosystem": "PyPI"}
    }
    try:
        response = await CLIENT.post(url, json=query)
        vulns: VulnerabilityList = response.json().get("vulns", []) if response.status_code == 200 else []
        VULN_CACHE[cache_key] = vulns
        return vulns
    except (httpx.RequestError, ValueError) as exc:
        logger.error("OSV query failed for %s: %s", cache_key, exc)
        return []


def get_best_version_fast(
    metadata: Optional[MetadataDict],
    constraint_str: str,
    before_date: datetime
) -> tuple[Optional[str], Optional[datetime]]:
    """
    Find the newest version matching constraints released before a specific date.

    :param metadata: Processed metadata dictionary.
    :param constraint_str: Version specifier (e.g., ">=1.0").
    :param before_date: Cut-off date for release.
    :return: A tuple of (version_string, release_date).
    """
    if not metadata or 'processed_releases' not in metadata:
        return None, None

    spec = None
    if constraint_str and constraint_str != "*":
        try:
            spec = SpecifierSet(constraint_str)
        except ValueError:
            return None, None

    # Reverse iterate to find the most recent matching version
    for ver_obj, rel_date, ver_str in reversed(metadata['processed_releases']):
        if rel_date and rel_date <= before_date:
            if spec is None or ver_obj in spec:
                return ver_str, rel_date
    return None, None


async def analyze_snapshot_async(row: dict[str, Any]) -> dict[str, Any]:
    """
    Analyze a single snapshot row for TTU and TTR metrics.

    :param row: A dictionary representing a row from the Polars DataFrame.
    :return: Dictionary containing calculated metrics.
    """
    pkg_name = row["package_name"]
    start_date = ensure_utc(row["package_published_at"])
    end_date = ensure_utc(row["next_snapshot"])

    # Fallback return values
    empty_res = {
        "package_name": pkg_name,
        "snapshot_start": row["package_published_at"],
        "snapshot_end": row["next_snapshot"],
        "avg_ttu": 0.0,
        "avg_ttr": 0.0
    }

    if not start_date or not end_date:
        return empty_res

    metadata = await fetch_pypi_metadata(pkg_name)
    if not metadata:
        return empty_res

    parent_ver, _ = get_best_version_fast(metadata, "*", start_date)
    if not parent_ver:
        return empty_res

    try:
        ver_url = f"https://pypi.org/pypi/{pkg_name}/{parent_ver}/json"
        ver_res = await CLIENT.get(ver_url)
        deps_list: list[str] = ver_res.json().get("info", {}).get("requires_dist", []) or []
    except (httpx.RequestError, ValueError):
        return empty_res

    ttu_sums: list[float] = []
    ttr_sums: list[float] = []

    for dep_raw in deps_list:
        try:
            req = Requirement(dep_raw)
            if req.marker and not req.marker.evaluate():
                continue

            dep_meta = await fetch_pypi_metadata(req.name)
            if not dep_meta:
                continue

            curr_v_str, curr_v_date = get_best_version_fast(
                dep_meta, str(req.specifier), start_date
            )
            if not curr_v_str or not curr_v_date:
                continue

            curr_v = pkg_version.parse(curr_v_str)

            # TTU: Time to Update
            outdated_days = 0.0
            for rel_v, rel_date, _ in dep_meta['processed_releases']:
                if rel_date and rel_v > curr_v and rel_date < end_date:
                    lag_start = max(start_date, rel_date)
                    outdated_days = max(
                        outdated_days,
                        (end_date - lag_start).total_seconds() / 86400
                    )
            ttu_sums.append(outdated_days)

            # TTR: Time to Remedy
            vulns = await get_vulnerabilities_osv(req.name, curr_v_str)
            max_ttr = 0.0
            for vuln in vulns:
                for affected in vuln.get("affected", []):
                    if affected.get("package", {}).get("name") != req.name:
                        continue
                    for v_range in affected.get("ranges", []):
                        for event in v_range.get("events", []):
                            if "fixed" in event:
                                _, fix_date = get_best_version_fast(
                                    dep_meta, f"=={event['fixed']}", end_date
                                )
                                if fix_date:
                                    remedy_at = max(start_date, fix_date)
                                    if remedy_at < end_date:
                                        ttr = (end_date - remedy_at).total_seconds() / 86400
                                        max_ttr = max(max_ttr, ttr)
            ttr_sums.append(max_ttr)
        except Exception as exc: # pylint: disable=broad-except
            logger.debug("Error processing dependency %s: %s", dep_raw, exc)
            continue

    return {
        **empty_res,
        "avg_ttu": sum(ttu_sums) / len(ttu_sums) if ttu_sums else 0.0,
        "avg_ttr": sum(ttr_sums) / len(ttr_sums) if ttr_sums else 0.0
    }


async def process_dataframe_async(df: pl.DataFrame) -> pl.DataFrame:
    """
    Main entry point to process a Polars DataFrame of snapshots.

    :param df: Input Polars DataFrame.
    :return: Resulting Polars DataFrame with metrics.
    """
    tasks = [analyze_snapshot_async(row) for row in df.to_dicts()]
    results = await asyncio.gather(*tasks)
    return pl.DataFrame(results)

result_df = asyncio.run(process_dataframe_async(snapshots_with_bounds))

RuntimeError: asyncio.run() cannot be called from a running event loop

In [ ]:
results_df.head()